#1. Install packages

In [1]:
!pip -q install -U pyarrow pandas huggingface_hub

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 20.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 110.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 793.2/793.2 kB 52.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.3, but you have pandas 3.0.5 which is incompatible.


#2. Imports and configuration

In [2]:
import os
import gc
import random
import pandas as pd

from datasets import load_dataset

# -----------------------------
# Configuration
# -----------------------------

DATASET_NAME = "McAuley-Lab/Amazon-Reviews-2023"

REVIEWS_PER_CATEGORY = 10000
SEED = 42

OUTPUT_DIR = "/content/buyproof_data"
OUTPUT_FILE = f"{OUTPUT_DIR}/buyproof.parquet"

os.makedirs(OUTPUT_DIR, exist_ok=True)

random.seed(SEED)

#3. Define all 33 categories

In [3]:
CATEGORIES = [
    "All_Beauty",
    "Amazon_Fashion",
    "Appliances",
    "Arts_Crafts_and_Sewing",
    "Automotive",
    "Baby_Products",
    "Beauty_and_Personal_Care",
    "Books",
    "CDs_and_Vinyl",
    "Cell_Phones_and_Accessories",
    "Clothing_Shoes_and_Jewelry",
    "Collectibles_and_Art",
    "Computers",
    "Electronics",
    "Garden_and_Outdoor",
    "Gift_Cards",
    "Grocery_and_Gourmet_Food",
    "Handmade_Products",
    "Health_and_Household",
    "Health_and_Personal_Care",
    "Home_and_Kitchen",
    "Industrial_and_Scientific",
    "Kindle_Store",
    "Movies_and_TV",
    "Musical_Instruments",
    "Office_Products",
    "Patio_Lawn_and_Garden",
    "Pet_Supplies",
    "Software",
    "Sports_and_Outdoors",
    "Subscription_Boxes",
    "Tools_and_Home_Improvement",
    "Toys_and_Games",
]

print("Number of categories:", len(CATEGORIES))

Number of categories: 33


#4. Function to sample 20K reviews

In [8]:
import requests

def sample_reviews(category, n=10000):
    print(f"\nLoading reviews: {category}")

    url = (
        "https://huggingface.co/datasets/McAuley-Lab/Amazon-Reviews-2023"
        f"/resolve/main/raw/review_categories/{category}.jsonl"
    )

    rows = []

    with requests.get(url, stream=True) as r:
        r.raise_for_status()
        for line in r.iter_lines(decode_unicode=True):
            if not line:
                continue

            try:
                row = json.loads(line)
            except json.JSONDecodeError:
                continue

            rows.append({
                "rating": row.get("rating"),
                "review_title": row.get("title"),
                "review_text": row.get("text"),
                "parent_asin": row.get("parent_asin"),
                "timestamp": row.get("timestamp"),
                "helpful_vote": row.get("helpful_vote"),
                "verified_purchase": row.get("verified_purchase"),
            })

            if len(rows) >= n:
                break

    df = pd.DataFrame(rows)
    df["category"] = category

    print(f"Collected {len(df):,} reviews")

    return df

#5. Function to get product metadata

In [9]:
import requests

def get_metadata(category, parent_asins, max_scan=500_000):
    print(f"Loading metadata: {category}")

    parent_asins = set(parent_asins)
    needed = len(parent_asins)

    url = (
        "https://huggingface.co/datasets/McAuley-Lab/Amazon-Reviews-2023"
        f"/resolve/main/raw/meta_categories/meta_{category}.jsonl"
    )

    rows = []
    scanned = 0

    with requests.get(url, stream=True) as r:
        r.raise_for_status()
        for line in r.iter_lines(decode_unicode=True):
            if not line:
                continue

            scanned += 1

            try:
                row = json.loads(line)
            except json.JSONDecodeError:
                continue

            if row.get("parent_asin") in parent_asins:
                rows.append({
                    "parent_asin": row.get("parent_asin"),
                    "product_title": row.get("title"),
                    "average_rating": row.get("average_rating"),
                    "rating_number": row.get("rating_number"),
                    "features": row.get("features"),
                    "store": row.get("store"),
                })

            if len(rows) >= needed or scanned >= max_scan:
                break

    df = pd.DataFrame(rows)

    print(f"Found metadata for {len(df):,} products (scanned {scanned:,} rows)")

    return df

#6. Process ONE category

In [12]:
import json

In [13]:
def process_category(category, n=10000):

    # Reviews
    reviews = sample_reviews(
        category=category,
        n=n
    )

    # Product metadata
    metadata = get_metadata(
        category=category,
        parent_asins=reviews["parent_asin"].unique()
    )

    # Merge
    merged = reviews.merge(
        metadata,
        on="parent_asin",
        how="left"
    )

    print(
        f"Final rows for {category}: "
        f"{len(merged):,}"
    )

    return merged

#7. Run all 33 categories

In [14]:
for i, category in enumerate(CATEGORIES, start=1):
    print("=" * 100)
    print(f"[{i}/{len(CATEGORIES)}] Processing {category}")
    print("=" * 100)

    try:
        df_category = process_category(category, n=REVIEWS_PER_CATEGORY)
        out_path = f"{OUTPUT_DIR}/{category}.parquet"
        df_category.to_parquet(out_path, index=False)
        print(f"Saved {len(df_category):,} rows -> {out_path}")

        # Free memory immediately, don't wait for next loop iteration.
        del df_category

    except Exception as e:
        print(f"ERROR in {category}: {e}")
        print("Skipping this category...")

    gc.collect()

[1/33] Processing All_Beauty

Loading reviews: All_Beauty
Collected 10,000 reviews
Loading metadata: All_Beauty
Found metadata for 7,067 products (scanned 101,891 rows)
Final rows for All_Beauty: 10,000
Saved 10,000 rows -> /content/buyproof_data/All_Beauty.parquet
[2/33] Processing Amazon_Fashion

Loading reviews: Amazon_Fashion
Collected 10,000 reviews
Loading metadata: Amazon_Fashion
Found metadata for 6,389 products (scanned 500,000 rows)
Final rows for Amazon_Fashion: 10,000
Saved 10,000 rows -> /content/buyproof_data/Amazon_Fashion.parquet
[3/33] Processing Appliances

Loading reviews: Appliances
Collected 10,000 reviews
Loading metadata: Appliances
Found metadata for 5,223 products (scanned 80,475 rows)
Final rows for Appliances: 10,000
Saved 10,000 rows -> /content/buyproof_data/Appliances.parquet
[4/33] Processing Arts_Crafts_and_Sewing

Loading reviews: Arts_Crafts_and_Sewing
Collected 10,000 reviews
Loading metadata: Arts_Crafts_and_Sewing
Found metadata for 6,732 products (

# Summary on Data

In [15]:
import glob

files = glob.glob(f"{OUTPUT_DIR}/*.parquet")

summary_rows = []

for f in sorted(files):
    df = pd.read_parquet(f)
    category = df["category"].iloc[0] if len(df) else f.split("/")[-1].replace(".parquet", "")

    total = len(df)
    matched = df["product_title"].notna().sum()
    unmatched = total - matched

    summary_rows.append({
        "category": category,
        "total_rows": total,
        "matched": matched,
        "unmatched": unmatched,
        "match_rate_%": round(100 * matched / total, 1) if total else 0,
    })

summary_df = pd.DataFrame(summary_rows).sort_values("category").reset_index(drop=True)

print(summary_df.to_string(index=False))

print("\n" + "=" * 60)
print(f"TOTAL rows: {summary_df['total_rows'].sum():,}")
print(f"TOTAL matched: {summary_df['matched'].sum():,}")
print(f"TOTAL unmatched: {summary_df['unmatched'].sum():,}")
print(f"Overall match rate: {round(100 * summary_df['matched'].sum() / summary_df['total_rows'].sum(), 1)}%")

summary_df.to_csv(f"{OUTPUT_DIR}/metadata_match_summary.csv", index=False)
print(f"\nSummary saved -> {OUTPUT_DIR}/metadata_match_summary.csv")

                   category  total_rows  matched  unmatched  match_rate_%
                 All_Beauty       10000    10000          0         100.0
             Amazon_Fashion       10000     7070       2930          70.7
                 Appliances       10000    10000          0         100.0
     Arts_Crafts_and_Sewing       10000     8086       1914          80.9
                 Automotive       10000     4333       5667          43.3
              Baby_Products       10000    10000          0         100.0
   Beauty_and_Personal_Care       10000     6902       3098          69.0
                      Books       10000     1495       8505          15.0
              CDs_and_Vinyl       10000     8928       1072          89.3
Cell_Phones_and_Accessories       10000     6081       3919          60.8
 Clothing_Shoes_and_Jewelry       10000     4682       5318          46.8
                Electronics       10000     5277       4723          52.8
                 Gift_Cards       1000

# Combining Whole Dataset in a single Parquet

In [16]:
files = glob.glob(f"{OUTPUT_DIR}/*.parquet")
combined = pd.concat([pd.read_parquet(f) for f in files], ignore_index=True)
combined.to_parquet(f"{OUTPUT_DIR}/all_categories.parquet", index=False)
print(f"Combined: {len(combined):,} total rows across {combined['category'].nunique()} categories")

Combined: 300,000 total rows across 30 categories
